# 02.1 · Enrich authors data

In [1]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv, find_dotenv

DATA = Path("../data")
CACHE = DATA / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
load_dotenv(dotenv_path=find_dotenv())                                   # tokens live in ../.env, never in the notebook

True

# ORCID Employments Crawler

Crawl ORCID iDs via the public API, extract employment history. Two parquet outputs: employments (one row per employment) and an index (one row per ORCID, the resume checkpoint).

**Key structural fact:** ORCID has no bulk endpoint — one request per iD. Speed comes from bounded parallelism, not batching.

---

## 📤 Output schema

### `orcid_employments.parquet` — one row per employment entry

| Column | Type | Notes |
|---|---|---|
| `orcid` | `str` | normalized iD |
| `put_code` | `int` | ORCID's internal ID; natural key with `orcid` |
| `organization` | `str` | employer name |
| `organization_address` | `str` | `"City, Region, CountryCode"` |
| `department` | `str` | `None` if absent |
| `role` | `str` | `None` if absent |
| `start_date` | `str` | raw ORCID partial date (`"2015"`, `"06/2019"`) |
| `end_date` | `str` | `""` or `None` = ongoing |
| `url` | `str` | `None` if absent |

Dates stored verbatim, no parsing.

### `orcid_employments_index.parquet` — one row per ORCID (checkpoint)

| Column | Type | Notes |
|---|---|---|
| `orcid` | `str` | normalized iD |
| `n_employments` | `int` | `0` is a cached, valid answer |
| `fetched_at` | `str` | ISO-8601 UTC |

**Why a separate index file:** if the checkpoint were "ORCIDs present in the employments file", anyone with zero employments would be re-queried every run. The index records "asked, answer was zero" explicitly.

---

## 🛑 Halt logic

Shared `threading.Event` as a circuit breaker. Any worker that sees a hard-stop signal sets it; every other worker checks it at three points (before start, after rate-limiter acquisition, inside backoff) and raises `CrawlHalted`. `stop.wait(t)` replaces `time.sleep(t)` so a halt interrupts in-flight backoff sleeps. `main` saves in a `finally:` block — **whatever was fetched is always persisted**.

| Signal | Action |
|---|---|
| `429` | **Halt** — daily quota exhausted |
| `401` | **Halt** — token rejected; every call would fail |
| `403` + "quota"/"rate"/"limit" in body | **Halt** |
| `503` | Retry with backoff (burst ceiling); **halt** after 5 attempts |
| `404` | Cache as `n_employments=0`, no retry |
| Other 4xx | Logged to `failed`, not cached, retried next run |
| Network / 5xx | Retry with backoff, then `CrawlHalted` |

Halted ORCIDs never reach `index.append(...)`, so they're **not** in the checkpoint and are re-queued automatically next run.

---

## 🔁 Resume invariant

**Only successfully-fetched entities enter the checkpoint.** Rerunning after a halt picks up exactly where it stopped — no duplicates, no re-fetches of cached ORCIDs (including zero-employment ones).

Dedupe on `(orcid, put_code)` keeps the **first** version seen. Same `put_code` with updated `end_date` in ORCID → old version wins. Change `keep="first"` → `keep="last"` in `save_results` for refresh semantics.

---

## 🔧 Config

| Variable | Default | Notes |
|---|---|---|
| `ORCID_USER_ID` | *required* | env var — public API client ID |
| `ORCID_USER_PASSWD` | *required* | env var — public API client secret |
| `MAX_WORKERS` | `4` | concurrent requests |
| `RATE_PER_SEC` | `10` | target req/s — limiter governs, not workers |
| `MAX_RETRIES` | `5` | attempts before halt on transient errors |

Credentials at [orcid.org/developer-tools](https://orcid.org/developer-tools). Anonymous: 25k reads/day. Registered public client: 100k/day. Hard ceiling is 12 req/s with a 40-request burst.

**Raising `MAX_WORKERS` without raising `RATE_PER_SEC` gains nothing** — extra workers just wait at the limiter.

---

## ⚠️ Gotchas that bite

- **Dates are strings.** `"2015"`, `"06/2019"`, `"2019-06-01"` can all appear. Parse before comparing.
- **`end_date` has two "ongoing" forms** — `""` and `None`. Use `emp["end_date"].fillna("") == ""`.
- **`organization_address` is unstructured.** No ROR ID in this endpoint. Deduplicating "Universitat de Girona" vs "University of Girona" requires a separate entity-resolution step (ROR lookup).
- **Checkpoint files are a pair.** Delete both or neither — an index without its employments file (or vice versa) leaves inconsistent state. To refresh one person, delete their index row; to refresh all, delete both files.
- **Append-only.** ORCIDs removed from the input stay in the output files.
- **Token expiry halts, doesn't limp.** A 401 mid-crawl stops immediately rather than producing N identical failures; rerunning with a fresh token resumes cleanly.
- **Python 3.9+** for `cancel_futures=True`. Older versions still work (queued ORCIDs run but return immediately since `fetch_employments` checks `stop` at entry).

In [ ]:
import os
import time
import threading
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from pyorcid import OrcidAuthentication, Orcid
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

# --- Config -----------------------------------------------------------------
MAX_WORKERS       = 4
RATE_PER_SEC      = 6
MAX_RETRIES       = 5
HTTP_TIMEOUT      = 20
CHECKPOINT_EVERY  = 50       # flush partial results every N processed ORCIDs

DATA = Path("../data")

EMPLOYMENT_COLS = [
    "orcid", "put_code",
    "organization", "organization_address",
    "department", "role",
    "start_date", "end_date", "url",
]


class CrawlHalted(RuntimeError):
    """Hard stop: daily quota spent, token rejected, or crawl aborted."""


# --- Ensure every outgoing HTTP call has a timeout --------------------------
# pyorcid doesn't expose a timeout. Patch requests.Session.request once,
# idempotently — re-running this cell must NOT re-wrap the function.
if not hasattr(requests, "_openalex_orig_session_request"):
    requests._openalex_orig_session_request = requests.Session.request

if not getattr(requests.Session.request, "_openalex_timeout_patched", False):
    _orig = requests._openalex_orig_session_request

    def _session_request_with_timeout(self, method, url, **kwargs):
        kwargs.setdefault("timeout", HTTP_TIMEOUT)
        return _orig(self, method, url, **kwargs)

    _session_request_with_timeout._openalex_timeout_patched = True
    requests.Session.request = _session_request_with_timeout


# --- Rate limiter (thread-safe token bucket) --------------------------------
class RateLimiter:
    def __init__(self, rate):
        self.min_interval = 1.0 / rate
        self.lock = threading.Lock()
        self.next_allowed = time.monotonic()

    def acquire(self):
        with self.lock:
            now = time.monotonic()
            wait = self.next_allowed - now
            if wait > 0:
                time.sleep(wait)
                now = time.monotonic()
            self.next_allowed = max(now, self.next_allowed) + self.min_interval


limiter = RateLimiter(RATE_PER_SEC)


# --- Helpers ----------------------------------------------------------------
def normalize_orcid(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    for p in ("https://orcid.org/", "http://orcid.org/",
              "https://www.orcid.org/", "http://www.orcid.org/", "orcid:"):
        if x.startswith(p):
            x = x[len(p):]
    x = x.strip().strip("/")
    parts = x.split("-")
    if len(parts) == 4 and all(len(p) == 4 for p in parts):
        return x
    return None


def _flatten(raw):
    """Normalize pyorcid's return shape (flat list or [[...]])."""
    if not raw:
        return []
    if isinstance(raw[0], list):
        return raw[0]
    return raw


def _halt_reason(exc):
    """Return a reason ONLY for genuine hard stops. Everything else → None."""
    resp   = getattr(exc, "response", None)
    status = getattr(resp, "status_code", None)
    text   = (str(exc) + " " + (getattr(resp, "text", "") or "")).lower()

    if status == 429:
        return "429 — daily quota exhausted"
    if status == 401:
        return "401 — token rejected/expired"
    if status == 403 and "quota" in text:
        return "403 — quota exhausted"
    return None


def fetch_employments(orcid_id, access_token, stop, attempt=0):
    """Return list-of-dicts on success, [] on 404, None if giving up.

    Raises CrawlHalted ONLY on hard stops (429/401/quota-403 or user abort).
    """
    if stop.is_set():
        raise CrawlHalted("stop already requested")

    limiter.acquire()
    if stop.is_set():
        raise CrawlHalted("stop during rate-limit wait")

    try:
        o = Orcid(orcid_id=orcid_id,
                  orcid_access_token=access_token,
                  state="public")
        return _flatten(o.employments())

    except Exception as e:
        status = getattr(getattr(e, "response", None), "status_code", None)

        # --- Hard stops ---
        reason = _halt_reason(e)
        if reason:
            stop.set()
            raise CrawlHalted(f"{orcid_id}: {reason}") from e

        # --- 404: no public record → cache as zero, never retry ---
        if status == 404:
            return []

        # --- Transient: retry with backoff, then GIVE UP (skip, don't halt) ---
        if attempt >= MAX_RETRIES:
            print(f"  ⚠ {orcid_id}: giving up after {MAX_RETRIES} retries "
                  f"({type(e).__name__}: {e})")
            return None

        if stop.wait(min(2 ** attempt, 30)):
            raise CrawlHalted("stop during backoff") from e
        return fetch_employments(orcid_id, access_token, stop, attempt + 1)


def to_rows(orcid_id, employments):
    return [{
        "orcid":                orcid_id,
        "put_code":             e.get("put-code"),
        "organization":         e.get("organization"),
        "organization_address": e.get("organization-address"),
        "department":           e.get("Department"),
        "role":                 e.get("Role"),
        "start_date":           e.get("start-date"),
        "end_date":             e.get("end-date"),
        "url":                  e.get("url"),
    } for e in employments]


# --- Crawl ------------------------------------------------------------------
def crawl(orcids, access_token,
          checkpoint_fn=None, checkpoint_every=CHECKPOINT_EVERY):
    """Returns (rows_df, index_df, failed, stopped, reason).

    Never raises on quota or user interrupt.
    If `checkpoint_fn(partial_rows_df, partial_index_df)` is provided, it is
    invoked every `checkpoint_every` successfully processed ORCIDs. The
    callback receives ONLY this run's accumulated data — the caller is
    responsible for merging with pre-existing state (save_results already
    does this). Callback exceptions are logged but do not abort the crawl.
    """
    stop  = threading.Event()
    rows, index, failed = [], [], []
    stopped, reason = False, None

    futures = {}
    pool = ThreadPoolExecutor(max_workers=MAX_WORKERS)
    try:
        futures = {pool.submit(fetch_employments, o, access_token, stop): o
                   for o in orcids}

        pbar = tqdm(as_completed(futures), total=len(futures),
                    desc="ORCIDs", unit="id", mininterval=0.5)
        try:
            for i, fut in enumerate(pbar, 1):
                orcid_id = futures[fut]
                try:
                    employments = fut.result()
                except CrawlHalted as e:
                    stopped, reason = True, (reason or str(e))
                    stop.set()
                    pbar.set_postfix_str("HALTED")
                    tqdm.write(f"  ⚠ HALTED — {reason}")
                    break
                except Exception as e:
                    failed.append((orcid_id, str(e)))
                    tqdm.write(f"  ✗ {orcid_id} failed: {e}")
                    continue

                if employments is None:
                    failed.append((orcid_id, "max retries exhausted"))
                    continue

                rows.extend(to_rows(orcid_id, employments))
                index.append({
                    "orcid":         orcid_id,
                    "n_employments": len(employments),
                    "fetched_at":    datetime.now(timezone.utc).isoformat(),
                })
                pbar.set_postfix_str(f"rows={len(rows)} failed={len(failed)}")

                # ---- Periodic checkpoint -----------------------------------
                if checkpoint_fn and i % checkpoint_every == 0:
                    try:
                        checkpoint_fn(pd.DataFrame(rows), pd.DataFrame(index))
                        tqdm.write(f"  💾 checkpoint at {i}/{len(futures)}  "
                                   f"({len(index)} ORCIDs, {len(rows)} rows)")
                    except Exception as ck_err:
                        tqdm.write(f"  ⚠ checkpoint failed at {i}: {ck_err}")

        except KeyboardInterrupt:
            stopped, reason = True, "user interrupt"
            stop.set()
            tqdm.write("  ⏹ interrupted — saving partial results")
    finally:
        stop.set()
        for f in futures:
            f.cancel()
        try:
            pool.shutdown(wait=True, cancel_futures=True)     # Python 3.9+
        except TypeError:
            pool.shutdown(wait=True)

    return (pd.DataFrame(rows), pd.DataFrame(index),
            failed, stopped, reason)


# --- Persistence helpers ----------------------------------------------------
def load_existing(employments_path, index_path):
    """Return (existing_rows_df, existing_index_df, already_set)."""
    if not index_path.exists():
        print(f"No previous {index_path.name} — starting fresh")
        return pd.DataFrame(), pd.DataFrame(), set()

    try:
        index    = pd.read_parquet(index_path)
        existing = (pd.read_parquet(employments_path)
                    if employments_path.exists() else pd.DataFrame())
    except Exception as e:
        raise RuntimeError(
            f"Found checkpoint files but could not read them ({e}). "
            f"Move or delete them to start fresh."
        ) from e

    already = set(index["orcid"].dropna()) if "orcid" in index.columns else set()
    print(f"Resuming: {len(already)} ORCIDs already crawled "
          f"({len(existing)} employment rows on disk)")
    return existing, index, already


def save_results(rows_df, index_df, existing_rows, existing_index,
                 employments_path, index_path, n_new, failed, verbose=True):
    combined_rows = pd.concat([existing_rows, rows_df], ignore_index=True)
    if not combined_rows.empty:
        combined_rows = (combined_rows
                         .drop_duplicates(subset=["orcid", "put_code"], keep="first")
                         .reindex(columns=EMPLOYMENT_COLS))

    combined_index = pd.concat([existing_index, index_df], ignore_index=True)
    if not combined_index.empty:
        combined_index = combined_index.drop_duplicates(subset="orcid", keep="first")

    combined_rows.to_parquet(employments_path, index=False, engine="pyarrow")
    combined_index.to_parquet(index_path, index=False, engine="pyarrow")

    if verbose:
        print(f"Wrote {len(combined_rows)} employment rows → "
              f"{employments_path.name} "
              f"(+{n_new} ORCIDs this run, {len(failed)} failures)")
        print(f"Wrote {len(combined_index)} ORCIDs → {index_path.name}")


# --- Main -------------------------------------------------------------------
if __name__ == "__main__":
    auth = OrcidAuthentication(
        client_id=os.environ["ORCID_USER_ID"],
        client_secret=os.environ["ORCID_USER_PASSWD"],
    )
    access_token = auth.get_public_access_token()

    df = pd.concat([
        pd.read_parquet(DATA / "cache/checked_orcids.parquet"),
        pd.read_parquet(DATA / "cache/unchecked_orcids.parquet"),
    ]).drop_duplicates()
    df["orcid_norm"] = df["orcid"].map(normalize_orcid)

    employments_path = DATA / "orcid_employments.parquet"
    index_path       = DATA / "orcid_employments_index.parquet"

    existing_rows, existing_index_df, already = load_existing(
        employments_path, index_path
    )

    todo = (df.loc[df["orcid_norm"].notna() &
                   ~df["orcid_norm"].isin(already), "orcid_norm"]
              .drop_duplicates()
              .tolist())

    print(f"{len(todo)} new ORCIDs to fetch "
          f"({len(df) - len(todo)} input rows already covered)")

    # Checkpoint callback: merges this run's partial data with the snapshot
    # taken at startup. save_results is idempotent, so repeated calls are safe.
    def checkpoint(partial_rows, partial_index):
        save_results(partial_rows, partial_index,
                     existing_rows, existing_index_df,
                     employments_path, index_path,
                     n_new=len(partial_index), failed=[],
                     verbose=False)

    new_rows = pd.DataFrame()
    new_index = pd.DataFrame()
    failed, stopped, reason = [], False, None

    try:
        if todo:
            new_rows, new_index, failed, stopped, reason = crawl(
                todo, access_token,
                checkpoint_fn=checkpoint,
                checkpoint_every=CHECKPOINT_EVERY,
            )
        else:
            print("Nothing new to fetch — skipping API calls")
    finally:
        # Final save: catches rows fetched after the last checkpoint boundary
        # and also writes the definitive summary (with the failed list).
        existing_index = (pd.read_parquet(index_path)
                          if index_path.exists() else pd.DataFrame())
        save_results(
            new_rows, new_index, existing_rows, existing_index,
            employments_path, index_path,
            n_new=len(new_index), failed=failed,
        )

        if failed:
            print(f"\n{len(failed)} ORCIDs gave up after retries — they are NOT "
                  f"in the index and will be retried next run:")
            for oid, err in failed[:10]:
                print(f"  {oid}: {err}")
            if len(failed) > 10:
                print(f"  … and {len(failed) - 10} more")

        if stopped:
            done    = len(new_index)
            pending = len(todo) - done - len(failed)
            kind    = ("user interrupt" if reason == "user interrupt"
                       else "ORCID quota/credential limit")
            print(f"\n⚠ STOPPED EARLY — {kind}.")
            print(f"  reason : {reason}")
            print(f"  saved  : {done} ORCIDs from this run "
                  f"({len(new_rows)} employment rows)")
            print(f"  pending: {pending} ORCIDs still un-crawled "
                  f"(they are NOT in the index)")
            print("  → rerun this cell to resume exactly where it stopped.")

In [3]:
pd.read_parquet('../data/orcid_employments.parquet')

,orcid,put_code,organization,organization_address,department,role,start_date,end_date,url
0,0000-0002-6245-7933,None,Universitat Rovira i Virgili,"Tarragona, Catalunya, ES","Electronic, Electrical and Automatic Control E...",Professor,11/1992,,NaN
1,0000-0001-6112-5913,None,Universitat Rovira i Virgili,"Tarragona, Catalunya, ES",Computer Science and Mathematics,Full Professor,09/1993,,http://deim.urv.cat/~francesc.serratosa/
2,0000-0001-5050-0167,None,Universitat de Lleida,"Lleida, Catalonia, ES",Animal Science,Lecturer,,,NaN
3,0000-0003-4161-5055,None,Universitat Pompeu Fabra,"Barcelona, Catalunya, ES",Economics,Professor,,,NaN
4,0000-0001-5434-7753,None,Ana M Gimenez Arnau,"SAN CUGAT DEL VALLES, Barcelona, ES",Dermatology,Dr,,,NaN
...,...,...,...,...,...,...,...,...,...
22563,0000-0002-8916-2086,None,AgroParisTech,"Paris, FR",SVS / UMR MoSAR [INRAe],Associate Professor,12/2023,,NaN
22564,0000-0003-2316-0210,None,Ghent University,"Ghent, BE",Economics,Postdoctoral researcher,11/2021,,NaN
22565,0000-0003-3679-7384,None,Universidad Andrs Bello,"Concepcin, concepcin/bobio, CL",Educacin y Ciencias Sociales,Docente,04/2011,,http://www.unab.cl
22566,0000-0002-5857-1899,None,Peking University,"Beijing, CN",College of Urban and Environmental Sciences,assistant professor,10/2020,,NaN
